# A service keeps its current graph in one file, and every merge carries it

The demo and Maxitor each print their graph, and ``gnomon write`` saves it in a file beside the
package — ``packages/aoa-demo/graph.json``, ``packages/aoa-maxitor/graph.json`` — listed in
``gnomon.toml`` at the repository root. The file changes in the same pull request as the code that
changed the graph, so git holds the graph's whole history; CI runs ``gnomon check`` and fails when a
file is stale. The format belongs to ``maxitor-gnomon``, a package that knows nothing about AOA.

The cases, with the real output at the bottom:

  1. the repository's config: which graphs, built how, saved where;
  2. a saved graph passes the schema of the version it declares;
  3. built again, the graph is the file byte for byte;
  4. a stale copy: ``check`` names what changed, by id — what the file holds → what the build
     prints — and writes nothing;
  5. a copy written with another indent: only the formatting differs;
  6. a document of an unknown version is refused by name.

Everything this example writes goes into a temporary folder.

## Import the package

In [1]:
from __future__ import annotations

import json
import os
import tempfile
from dataclasses import replace
from pathlib import Path

from maxitor_gnomon.config import read_config
from maxitor_gnomon.schemas import GraphRefused, check_graph
from maxitor_gnomon.writer import build, check, render

## Work from the repository root

In [2]:
# The repository root: the folder that holds gnomon.toml, wherever this runs from.
REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "gnomon.toml").is_file())
os.chdir(REPO_ROOT)

## 1. The repository's graphs: built how, saved where

In [3]:
print("1. The repository's graphs")
GRAPHS = read_config("gnomon.toml")
for graph in GRAPHS:
    print(f"   {graph.name:<8} {graph.build}  →  {graph.file.relative_to(REPO_ROOT)}")
DEMO = next(graph for graph in GRAPHS if graph.name == "demo")

1. The repository's graphs
   demo     uv run --extra dev python -m aoa.demo.print_graph  →  packages/aoa-demo/graph.json
   maxitor  uv run --extra dev python -m aoa.maxitor.print_graph  →  packages/aoa-maxitor/graph.json


## 2. A saved graph against the schema of its version

In [4]:
print("2. A saved graph against its schema")
saved = json.loads(DEMO.file.read_text(encoding="utf-8"))
check_graph(saved)
print(f"   {DEMO.file.relative_to(REPO_ROOT)}: {saved['schema_version']},"
      f" {len(saved['nodes'])} nodes, {len(saved['edges'])} edges — a graph")

2. A saved graph against its schema
   packages/aoa-demo/graph.json: 1.1, 3213 nodes, 4841 edges — a graph


## 3. Built again, the graph is the file byte for byte

In [5]:
print("3. Built again")
print("   the same bytes as the file:", render(build(DEMO)) == DEMO.file.read_bytes())

3. Built again
   the same bytes as the file: True


## 4–5. A stale copy and a copy with another indent, in a temporary folder

In [6]:
with tempfile.TemporaryDirectory() as folder:
    copy = replace(DEMO, file=Path(folder) / "graph.json")

    print("4. A stale copy")
    stale = json.loads(json.dumps(saved))
    node = next(node for node in stale["nodes"] if node["id"].endswith(".StoreDomain"))
    node["label"] = "Shop"
    stale["edges"].pop()
    copy.file.write_bytes(render(stale))
    (outcome,) = check([copy])
    print("  ", outcome.state)
    for line in outcome.details:
        print("    ", line)

    print("5. Another indent")
    copy.file.write_text(json.dumps(saved, indent=4), encoding="utf-8")
    (outcome,) = check([copy])
    print(f"   {outcome.state}: {outcome.details[0]}")

4. A stale copy
   stale
     edges[id=aoa.demo.model.telemetry_pipeline.telemetry_pipeline_domain.TelemetryPipelineDomain|application|aoa.action_machine.application.application.Application]: added
     nodes[id=aoa.demo.model.store.store_domain.StoreDomain].label: Shop → StoreDomain
5. Another indent
   stale: only the formatting differs


## 6. An unknown version is refused by name

In [7]:
print("6. An unknown version")
try:
    check_graph({**saved, "schema_version": "9.9"})
except GraphRefused as refused:
    print(f"   GraphRefused: {refused}")

6. An unknown version
   GraphRefused: graph 9.9: unknown schema version '9.9'; known: 1.0, 1.1
